# 2-motor: no-load-current

Measure the current the free motor draws against its speed, the drag that current stands for, and the inertia that ties the spin's current to the coast's deceleration, all from the bemf-coast dataset.

## Purpose

A motor turning with nothing on its shaft still draws current. That current is the torque the motor spends on its own drag, the brushes on the commutator, the bearings, the air around the rotor and the iron loss in the rotor's core, divided by the torque constant. Every current the servo reads under motion carries it, so a stall detector, a load estimate or a torque command that does not know the no-load current reads the motor's own drag as load. The no-load current rises with speed, and the shape of the rise says what kind of drag it is.

This experiment reads it off the spins of the bemf-coast dataset, where the free motor is driven at a set duty for a second and a half and then let go. The last stretch of each spin gives the current at a settled speed, the ripple in the same current gives the speed, and the coast that follows gives the drag a second way, as the rate the rotor slows at. The two routes are tied by the rotor's inertia. At steady speed the drive's torque equals the drag, and in the coast the drag alone slows the rotor, so one inertia should turn the coast's deceleration into the spin's current at the same speed. Whether one inertia fits every rung at both levels is the test of whether the current's rise is the same drag the coast shows, or carries a loss only the drive has. Stage 3 reads the servo's friction and inertia in situ through the gear train, and the motor's own share is graded against the numbers here.

## Procedure

This notebook has no capture of its own. It reads the dataset that `capture.py` of the bemf-coast experiment writes, so the procedure is that script's. The bare motor sits in the motor rig with the wrench bar off and its leads on the board's motor connector, the board on the DPS with the adapter on its bus at 3 Mbps, running the experiment image. The script trims the rail to 5.0 V and then to 7.4 V, and at each level runs cycles at 15, 20, 30, 40 and 60 % duty, three per duty. One cycle is one `osc sweep`, half a second of torque-off baseline, then the spin at the duty for 1.5 s, then 0.8 s with the bridge open while the rotor coasts, and the same spin and coast the other way. Twenty seconds of rest follow every cycle, the bench rule for a bare motor, so the winding stays within a degree or two of the can. Thirty cycles in all, sixty spins and sixty coasts.

The spin is a second and a half because the earlier bare-motor work found 300 ms too short, with the rotor still speeding up at the end of the window. Whether 1.5 s is long enough is one of the things measured here, from the shape of each spin's tail.

## Data

- `cycles.csv`: one row per cycle. `k` the cycle number, `volts_set` the rail level asked for, `dps_read` the DPS output that reached it, `vsys_read` the rail the board read at rest before the level's first cycle, `duty_pct`, `repeat`, `t_start` and `t_end` the host time the sweep ran, `folder` the cycle's folder.
- `cycles/<k>/sweep.csv`, `meta.json`: one row per control tick at 20 kHz. Segment 0 is the torque-off baseline. Segments 1 and 2 are the spin and the coast one way, 3 and 4 the same the other way. A coast segment carries a `cmd_duty_q15` of 0, and its first row or two still carry the spin's `duty_counts` until the open-bridge goal lands, so a coast starts at its first row with `duty_counts` of 0. `tick` restarts at 0 in every segment, so the time the host takes between the spin's last row and the coast's first is not in the file. `current_raw`, `vmotor_a`, `vmotor_b`, `vbus_raw` and `ntc_raw` are ADC counts. `vref_raw` is in the header and empty, since the row's telemetry chunk does not carry it.
- The bemf-coast result, `../bemf-coast/results/<motor>.json`, read through `oscnb.results`. This notebook assumes two keys there. `ke_v_s_per_rad` is the back-EMF constant with its bounds, and `cycles` is a list with one entry per spin, each with `k`, `level`, `duty`, `direction`, `speed_rev_s`, `emf_v`, `i_spin_a` and `ke`. The speed there is counted in the coasting terminal voltage, the one here in the spin's current, and the two are compared below.
- The stage 1 results for the board, current-scale for the amps and terminal-chain for the volts, from `stages/1-board/*/results/<board>.json`. The tap difference is converted through `terminal_counts_per_v` alone, on the reading that the tap offset is the same on both taps and drops out of the difference.

### cycles.csv

Every cycle of the capture, and how far the trim landed from each level.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import butter, sosfiltfilt

from oscnb import boards, laps, motors, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"   # the house colors
MOTOR = motors.MOTORS[motors.DEFAULT]                                   # the motor, with its counted parts
BOARD = boards.BOARDS[boards.DEFAULT]                                   # the board, with the parts' nominal values
CAPTURE = (Path("../../../telemetry") / f"{MOTOR.key}__{BOARD.key}" / "bemf-coast" / "capture-1").resolve()   # the sibling's dataset
FS = BOARD.tick_hz                                                      # rows per second in a sweep
print(MOTOR.key, "on", BOARD.key, "from", results.rel(CAPTURE))
print(f"{MOTOR.label}: {MOTOR.slots} slots x {MOTOR.brushes} brushes = {MOTOR.ripple_per_rev} commutation ripples per revolution")

In [ ]:
cycles = pd.read_csv(CAPTURE / "cycles.csv").set_index("k")            # one row per cycle, the folder of its sweep
LEVELS = sorted(cycles["volts_set"].unique())                          # the rail levels, low to high
DUTIES = sorted(cycles["duty_pct"].unique())
COLOR = dict(zip(LEVELS, [BLUE, ORANGE]))                               # one color per level, fixed through the notebook
cycles["trim_mv"] = (cycles["vsys_read"] - cycles["volts_set"]) * 1e3   # how far the trim landed from the level
show(cycles, n=None)
fig, ax = plt.subplots(figsize=(8, 3))
for L in LEVELS:
    g = cycles[cycles["volts_set"] == L]
    ax.plot(g.index, g["trim_mv"], "o", color=COLOR[L], label=f"{L:g} V")   # one dot per cycle
ax.axhline(0, color=INK, lw=0.5)
ax.set_xlabel("cycle k"); ax.set_ylabel("rail read at rest, less the level (mV)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"{len(cycles)} cycles, {len(LEVELS)} levels x {len(DUTIES)} duties x {cycles['repeat'].nunique()} repeats")

One row per cycle, three per duty at each level. The rail read before each level's first cycle is what the trim reached, and it should sit within the trim's 30 mV of the level.

### cycles/K/sweep.csv and meta.json

One cycle, the first 40 % cycle at the lower level. The current channel, the terminal taps and the applied window through the baseline, the spin and the coast each way.

In [ ]:
def folder_of(k):
    return CAPTURE / cycles.loc[k, "folder"]

K0 = int(cycles[(cycles["volts_set"] == LEVELS[0]) & (cycles["duty_pct"] == 40)].index[0])   # the cycle the figures walk through
df, meta = tel.load(folder_of(K0), BOARD)                 # checks the sense block against the board entry
print("cycle", K0, "| segments:", df["seg"].value_counts().sort_index().to_dict(), "| schedule:", meta["schedule"],
      "| dirs:", meta["dirs"], "| baseline ms:", meta["baseline_ms"], "| decay:", meta["decay"])
ts = np.arange(len(df)) / FS                              # time through the file; `t` restarts at each segment
coast_rows = df["seg"].gt(0) & df["cmd_duty_q15"].eq(0)   # a coast is a driven segment with a zero command
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax[0].plot(ts, df["current_raw"], lw=0.4, color=BLUE); ax[0].set_ylabel("current_raw (counts)")
ax[1].plot(ts, df["vmotor_a"], lw=0.4, color=BLUE, label="tap A"); ax[1].plot(ts, df["vmotor_b"], lw=0.4, color=ORANGE, label="tap B")
ax[1].set_ylabel("terminal taps (counts)"); ax[1].legend(loc="upper right")
ax[2].plot(ts, BOARD.duty_pct_of_ticks(df["duty_counts"]), lw=0.8, color=INK); ax[2].set_ylabel("applied window (%)")
for a in ax:
    a.fill_between(ts, 0, 1, where=coast_rows, transform=a.get_xaxis_transform(), color=MUTED, alpha=0.15)   # the coasts shaded
    a.grid(alpha=0.3)
ax[0].set_title(f"cycle {K0}: {cycles.loc[K0, 'volts_set']:g} V, {cycles.loc[K0, 'duty_pct']} % duty, the coasts shaded")
ax[-1].set_xlabel("time through the file (s)")
plt.tight_layout(); plt.show()
missing = {int(k): int(tel.gaps(tel.load(folder_of(k))[0]).sum()) for k in cycles.index}
print("missing rows per cycle:", {k: m for k, m in missing.items() if m} or "none")

The baseline at the zero, then the spin each way with the current settling as the rotor comes up to speed, then the coast with the taps carrying the back-EMF down to the bias node while the current channel sits at its zero. The taps swap with the direction, and the back-EMF keeps the drive's polarity.

### The results carried in

The back-EMF constant and the per-spin list from bemf-coast, the current chain from current-scale and the terminal and rail chains from terminal-chain.

In [ ]:
KE = results.read("2-motor", "bemf-coast", pair=MOTOR.key)            # Ke with its bounds, and one entry per spin
CS = results.read("1-board", "current-scale", pair=BOARD.key)         # counts per amp and the in-drive offset
TC = results.read("1-board", "terminal-chain", pair=BOARD.key)        # counts per volt at the taps and at the rail tap
KE_V = KE["ke_v_s_per_rad"]["value"]                                  # V s / rad, the same number as N m / A
COUNTS_PER_A = CS["counts_per_a"]["value"]
DRIVE_OFFSET = CS["drive_offset_counts"]["value"]                     # counts the chain reads over its zero while the bridge drives
TERM_CPV = TC["terminal_counts_per_v"]["value"]                       # counts per volt of tap difference
RAIL_CPV, RAIL_OFF = TC["rail_counts_per_v"]["value"], TC["rail_offset_counts"]["value"]
ke_cycles = pd.DataFrame(KE["cycles"])                                 # bemf-coast's own reading of every spin
print(f"Ke {KE_V * 1e3:.3f} mV s/rad ({KE['ke_v_s_per_rad']['lo'] * 1e3:.3f} to {KE['ke_v_s_per_rad']['hi'] * 1e3:.3f}), "
      f"{KE_V * 2 * np.pi * 1e3:.2f} mV per rev/s, from {results.rel(KE['capture'])}")
print(f"current chain {COUNTS_PER_A:.1f} counts/A, in-drive offset {DRIVE_OFFSET:+.1f} counts; taps {TERM_CPV:.1f} counts/V; rail {RAIL_CPV:.1f} counts/V, offset {RAIL_OFF:+.1f} counts")
show(ke_cycles, n=None)

The numbers every cycle is converted through, and bemf-coast's table of the same spins, one row each.

## Method

Each cycle is reduced the same way, both directions.

**The settled current.** The spin's current in amps is `current_raw` less the cycle's own torque-off zero, the median of the second half of the baseline, less the in-drive offset from current-scale, over that experiment's counts per amp. The window is the last 300 ms of the spin and the settled current is its mean. Whether the window is settled is read from the tail. A plain exponential

$$i(t) = i_\infty + A\,e^{-(t - t_1)/\tau_i}$$

fitted from 0.3 s into the spin to its end gives where the current is heading, $i_\infty$, and how fast it gets there, $\tau_i$. The gap between $i_\infty$ and the window's mean is what the window would still lose with a longer spin, and a window counts as settled when that gap is under one milliamp.

**The speed.** The current ripples once per commutation, six times per revolution on this motor, three slots and two brushes, `ripple_per_rev` in the registry. The strongest line of the window's spectrum between 100 and 4000 Hz seeds a band-pass of the same window, and the upward zero crossings of the filtered trace are counted over a whole number of ripples, which gives the ripple frequency to a fraction of a cycle. The speed is that frequency over six, in revolutions per second, and $\omega$ (omega, the angular speed) is $2\pi$ times it in radians per second. The spin's back-EMF at that speed is $E = K_e\,\omega$ with $K_e$ from bemf-coast.

**The coast's drag.** With the bridge open the terminal voltage is the back-EMF, read as the tap difference signed by the direction over the terminal chain's counts per volt, and it decays as the rotor slows. A viscous drag plus a constant one give

$$\frac{dE}{dt} = -a - \frac{E}{\tau}$$

where $a$ in volts per second is the Coulomb term, the drag that does not depend on speed, and $\tau$ (tau) in seconds is the viscous one, the drag in step with speed. Its solution, $E(t) = (E_0 + a\tau)\,e^{-t/\tau} - a\tau$ clipped at zero, is fitted through the whole coast down to where the EMF reaches 3 % of its start. The deceleration at any EMF is then $a + E/\tau$.

**The inertia.** At a settled speed the drive's torque, $K_t I$, equals the drag. In the coast the same drag slows the rotor, $J\,d\omega/dt = (J/K_e)\,dE/dt$. With $K_t = K_e$ in SI units, one inertia ties the two,

$$J = \frac{K_e^2\, I}{\left. dE/dt\right|_{E}}$$

with $I$ the window's current and the deceleration taken from the coast's fit at the spin's own EMF. A $J$ that comes out the same at every rung and level means the current's rise with speed is the drag the coast shows. A $J$ that climbs with speed means the drive carries a loss the free coast does not, iron loss under the chopped drive or a drive-only brush term, and the no-load current is then more than the coast's drag.

### The estimators

One function per step, applied to every spin and coast below.

In [ ]:
WINDOW_S = 0.3                      # the last stretch of the spin the current and the speed are read over
TAIL_FROM_S = 0.3                   # the tail fit starts here, past the spin's first transient
TAIL_TOL_MA = 1.0                   # a window counts as settled when the tail's asymptote sits within this of its mean
RIPPLE_BAND_HZ = (100.0, 4000.0)    # where the commutation line is looked for
TURN_V = 0.05                       # a coast whose first millisecond of EMF reads under this never turned
COAST_END = 0.03                    # the drag fit runs down to this share of the coast's starting EMF
WIN = int(WINDOW_S * FS)
RIPPLES = MOTOR.ripple_per_rev


def amps(raw, zero):
    # counts over the cycle's own rest zero, less the in-drive offset, through the graded scale
    return (np.asarray(raw, float) - zero - DRIVE_OFFSET) / COUNTS_PER_A


def emf(seg, d):
    # the tap difference signed by the direction, in volts; the bridge is open so this is the back-EMF
    return d * (seg["vmotor_a"].to_numpy(float) - seg["vmotor_b"].to_numpy(float)) / TERM_CPV


def spins_and_coasts(df):
    # (direction, spin rows, coast rows from its first open-bridge row, rows the goal took to land) per direction
    out, spin = [], None
    for _, g in df[df["seg"] > 0].groupby("seg"):
        if int(g["cmd_duty_q15"].iloc[0]) != 0:
            spin = g                                                   # a driven segment, the coast follows it
        elif spin is not None:
            z = np.flatnonzero(g["duty_counts"].to_numpy() == 0)        # the coast starts where the window is gone
            first = int(z[0]) if len(z) else len(g)
            out.append((int(spin["dir"].iloc[0]), spin, g.iloc[first:], first))
            spin = None
    return out


def line_freq(x, lo=RIPPLE_BAND_HZ[0], hi=RIPPLE_BAND_HZ[1]):
    # (frequency, prominence) of the strongest line in the band: a Hann-windowed spectrum padded 8x, its peak over the band's median
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    X = np.abs(np.fft.rfft(x * np.hanning(n), 8 * n))
    f = np.fft.rfftfreq(8 * n, 1 / FS)
    m = (f >= lo) & (f <= hi)
    k = np.argmax(X[m])
    return float(f[m][k]), float(X[m][k] / np.median(X[m]))


def count_ripples(x, f0, frac=0.35):
    # (mean frequency, whole ripples counted, the filtered trace): a 2nd-order band-pass around f0 run forward and back,
    # upward zero crossings interpolated to a fraction of a sample, the first and last crossing dropped as filter edge
    sos = butter(2, [f0 * (1 - frac), f0 * (1 + frac)], btype="band", fs=FS, output="sos")
    y = sosfiltfilt(sos, np.asarray(x, float) - np.mean(x))
    s = np.flatnonzero((y[:-1] < 0) & (y[1:] >= 0))
    if len(s) < 4:
        return np.nan, 0, y
    tc = (s + (-y[s]) / (y[s + 1] - y[s]))[1:-1]
    return (len(tc) - 1) / ((tc[-1] - tc[0]) / FS), len(tc) - 1, y


def tail_fit(t, i, t1=TAIL_FROM_S):
    # the spin's tail as an exponential approach from t1 on: (asymptote, amplitude, time constant), NaN when the fit fails
    m = t >= t1
    try:
        p, _ = curve_fit(lambda x, a, A, tau: a + A * np.exp(-(x - t1) / tau), t[m], i[m],
                         p0=[i[-WIN:].mean(), i[m][:WIN].mean() - i[-WIN:].mean(), 0.2], bounds=([-1, -2, 0.005], [2, 2, 5]))
        return tuple(float(v) for v in p)
    except (RuntimeError, ValueError):
        return (np.nan, np.nan, np.nan)


def drag_fit(t, E, e0_guess):
    # viscous plus Coulomb through the coast down to COAST_END of its start: (E0, tau, a), NaN when the fit fails
    model = lambda x, e0, tau, a: np.maximum((e0 + a * tau) * np.exp(-x / tau) - a * tau, 0.0)
    below = np.flatnonzero(E < COAST_END * e0_guess)
    stop = int(below[0]) if len(below) else len(E)
    try:
        p, _ = curve_fit(model, t[:stop], E[:stop], p0=[e0_guess, 0.2, 5.0], bounds=([0, 1e-3, 0], [20, 50, 500]))
        return tuple(float(v) for v in p)
    except (RuntimeError, ValueError):
        return (np.nan, np.nan, np.nan)


def reduce_cycle(k):
    # every estimator over one cycle: one row per direction
    df, meta = tel.load(folder_of(k), BOARD)
    zero = float(tel.baseline(df)["current_raw"].median())              # the cycle's own torque-off zero
    rows = []
    for d, spin, coast, landed in spins_and_coasts(df):
        t = np.arange(len(spin)) / FS
        i = amps(spin["current_raw"], zero)
        w = i[-WIN:]                                                     # the window: the spin's last 300 ms
        i_inf, _, tau_i = tail_fit(t, i)
        f_pk, prom = line_freq(spin["current_raw"].to_numpy(float)[-WIN:])
        f_rip, n_rip, _ = count_ripples(spin["current_raw"].to_numpy(float)[-WIN:], f_pk)
        speed = f_rip / RIPPLES                                          # rev/s at the motor shaft
        omega = 2 * np.pi * speed
        row = {"k": k, "level": float(cycles.loc[k, "volts_set"]), "duty": int(cycles.loc[k, "duty_pct"]), "dir": d,
               "zero": zero, "duty_applied_pct": float(BOARD.duty_pct_of_ticks(spin["duty_counts"].abs().iloc[-WIN:].median())),
               "rail_v": float((spin["vbus_raw"].iloc[-WIN:].mean() - RAIL_OFF) / RAIL_CPV),
               "ntc_c": float(BOARD.ntc_c_beta(spin["ntc_raw"].iloc[-WIN:].mean())),
               "i_win_a": float(w.mean()), "i_sd_a": float(w.std()), "i_inf_a": i_inf, "tail_tau_s": tau_i,
               "tail_gap_ma": (i_inf - w.mean()) * 1e3,                  # what a longer spin would still take off the window
               "f_ripple_hz": f_rip, "ripples": n_rip, "line_prominence": prom,
               "speed_rev_s": speed, "e_spin_v": KE_V * omega,            # the spin's back-EMF at the counted speed
               "coast_landed_rows": landed, "coast_rows": len(coast)}
        if len(coast):
            E = emf(coast, d)
            tc = np.arange(len(E)) / FS
            row["e_first_v"] = float(E[2:20].mean())                     # 0.1 to 1 ms into the coast
            row["turned"] = row["e_first_v"] >= TURN_V
            if row["turned"]:
                e0, tau, a = drag_fit(tc, E, row["e_first_v"])
                dec = a + row["e_spin_v"] / tau                          # the coast's deceleration at the spin's EMF, V/s
                row.update({"coast_e0_v": e0, "drag_tau_s": tau, "drag_a_v_per_s": a, "dec_at_spin_v_per_s": dec,
                            "j_kg_m2": KE_V ** 2 * row["i_win_a"] / dec})
        else:
            row["turned"] = False
        rows.append(row)
    return rows

### One cycle through the estimators

The same cycle as above, forward. The spin's current with the window and the tail fit, the band-passed ripple at the start of the window, and the coast's back-EMF with the drag fit.

In [ ]:
d0, spin0, coast0, landed0 = spins_and_coasts(df)[0]                   # the forward spin and coast of cycle K0
zero0 = float(tel.baseline(df)["current_raw"].median())
t0 = np.arange(len(spin0)) / FS
i0 = amps(spin0["current_raw"], zero0)
i_inf0, A0, tau0 = tail_fit(t0, i0)
f_pk0, prom0 = line_freq(spin0["current_raw"].to_numpy(float)[-WIN:])
f_rip0, n_rip0, y0 = count_ripples(spin0["current_raw"].to_numpy(float)[-WIN:], f_pk0)
E0 = emf(coast0, d0); tc0 = np.arange(len(E0)) / FS
e_first0 = float(E0[2:20].mean())
e00, dtau0, da0 = drag_fit(tc0, E0, e_first0)
fit_E = np.maximum((e00 + da0 * dtau0) * np.exp(-tc0 / dtau0) - da0 * dtau0, 0.0)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(t0, i0 * 1e3, lw=0.3, color=BLUE, label="current")
m = t0 >= TAIL_FROM_S
ax[0].plot(t0[m], (i_inf0 + A0 * np.exp(-(t0[m] - TAIL_FROM_S) / tau0)) * 1e3, "-", color=INK, lw=1.2,
           label=f"tail: {i_inf0 * 1e3:.1f} mA, tau {tau0 * 1e3:.0f} ms")
ax[0].axvspan(t0[-WIN], t0[-1], color=MUTED, alpha=0.2, label="the window")
ax[0].axhline(i_inf0 * 1e3, color=INK, lw=0.5, ls=":")
ax[0].set_xlabel("time in the spin (s)"); ax[0].set_ylabel("current (mA)"); ax[0].set_title("the spin, its tail fit and the window"); ax[0].legend(fontsize=8)
tw = np.arange(WIN) / FS * 1e3
show_ms = 20
ax[1].plot(tw[tw < show_ms], (spin0["current_raw"].to_numpy(float)[-WIN:] - spin0["current_raw"].iloc[-WIN:].mean())[tw < show_ms], lw=0.6, color=BLUE, label="current, window mean removed")
ax[1].plot(tw[tw < show_ms], y0[tw < show_ms], lw=1.2, color=ORANGE, label=f"band-passed at {f_pk0:.0f} Hz")
ax[1].axhline(0, color=INK, lw=0.5)
ax[1].set_xlabel("time into the window (ms)"); ax[1].set_ylabel("counts"); ax[1].set_title(f"the ripple: {f_rip0:.1f} Hz over {n_rip0} ripples = {f_rip0 / RIPPLES:.1f} rev/s"); ax[1].legend(fontsize=8)
ax[2].plot(tc0 * 1e3, E0, lw=0.4, color=BLUE, label="back-EMF")
ax[2].plot(tc0 * 1e3, fit_E, "-", color=INK, lw=1.2, label=f"fit: tau {dtau0 * 1e3:.0f} ms, a {da0:.2f} V/s")
ax[2].set_xlabel("time in the coast (ms)"); ax[2].set_ylabel("terminal volts (V)"); ax[2].set_title("the coast and its drag fit"); ax[2].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"cycle {K0} forward: window {i0[-WIN:].mean() * 1e3:.1f} mA (sd {i0[-WIN:].std() * 1e3:.1f}), asymptote {i_inf0 * 1e3:.1f} mA, gap {(i_inf0 - i0[-WIN:].mean()) * 1e3:+.2f} mA; "
      f"line at {f_pk0:.0f} Hz with prominence {prom0:.0f}; coast goal landed after {landed0} rows, EMF at 0.1 to 1 ms {e_first0:.3f} V, "
      f"spin EMF {KE_V * 2 * np.pi * f_rip0 / RIPPLES:.3f} V at the counted speed")

Left, the spin's current from the start, the exponential through its tail and the window shaded, with the asymptote dotted. Middle, the first 20 ms of the window with the band-passed ripple over the raw counts, the crossings of the orange trace being what is counted. Right, the coast's back-EMF from the first open-bridge row with the viscous plus Coulomb curve through it.

### Every cycle

Every spin and coast through the same estimators, one row per direction.

In [ ]:
CYC = pd.DataFrame([r for k in cycles.index for r in reduce_cycle(k)])
CYC["turned"] = CYC["turned"].fillna(False).astype(bool)
CYC = CYC.merge(ke_cycles[["k", "direction", "speed_rev_s", "emf_v", "i_spin_a"]].rename(
    columns={"direction": "dir", "speed_rev_s": "speed_bemf_rev_s", "emf_v": "emf_bemf_v", "i_spin_a": "i_bemf_a"}), on=["k", "dir"], how="left")
CYC["speed_over_bemf_pct"] = (CYC["speed_rev_s"] / CYC["speed_bemf_rev_s"] - 1) * 100     # the spin's ripple speed against the coast's
CYC["settled"] = CYC["tail_gap_ma"].abs() < TAIL_TOL_MA
show(CYC[["k", "level", "duty", "dir", "duty_applied_pct", "rail_v", "i_win_a", "i_sd_a", "i_inf_a", "tail_tau_s", "tail_gap_ma", "settled",
          "f_ripple_hz", "ripples", "speed_rev_s", "speed_bemf_rev_s", "e_spin_v", "turned", "drag_tau_s", "drag_a_v_per_s", "dec_at_spin_v_per_s", "j_kg_m2"]], n=None)
print(f"{len(CYC)} spins, {int(CYC['turned'].sum())} coasts turned, {int(CYC['settled'].sum())} windows settled by the {TAIL_TOL_MA:g} mA rule, "
      f"{int(CYC['j_kg_m2'].notna().sum())} inertias; the coast goal landed after {CYC['coast_landed_rows'].min()} to {CYC['coast_landed_rows'].max()} rows")

One row per spin. A coast that left no back-EMF never turned and has no drag fit, and a spin whose tail still runs at the end of the window is marked unsettled.

## Results

### Did the spins settle

The tail's asymptote less the window's mean per spin against speed, and the spin's ripple speed against the speed bemf-coast counted in the coast's own voltage. The thing to see is whether the gaps sit inside the tolerance at every rung, and whether the two speeds agree to about a percent, with the coast's a little lower since it comes after the window.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
MARK = {1: "o", -1: "^"}                                                   # the direction as the marker shape
for L in LEVELS:
    for d in (1, -1):
        g = CYC[(CYC["level"] == L) & (CYC["dir"] == d)]
        kw = dict(ls="none", marker=MARK[d], ms=6, color=COLOR[L], mfc=COLOR[L] if d == 1 else "none", label=f"{L:g} V, direction {d:+d}")
        ax[0].plot(g["speed_rev_s"], g["tail_gap_ma"], **kw)
        ax[1].plot(g["speed_rev_s"], g["speed_over_bemf_pct"], **kw)
ax[0].axhspan(-TAIL_TOL_MA, TAIL_TOL_MA, color=MUTED, alpha=0.15)
ax[0].set_ylabel("tail asymptote less window mean (mA)"); ax[0].set_title("what a longer spin would still change, the tolerance shaded")
ax[1].set_ylabel("ripple speed over bemf-coast's speed, less one (%)"); ax[1].set_title("the spin's speed against the coast's")
for a in ax:
    a.axhline(0, color=INK, lw=0.5); a.set_xlabel("speed from the window's ripple (rev/s)"); a.grid(alpha=0.3)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"tail gap: mean {CYC['tail_gap_ma'].mean():+.2f} mA, worst {CYC.loc[CYC['tail_gap_ma'].abs().idxmax(), 'tail_gap_ma']:+.2f} mA; "
      f"tail time constant {CYC['tail_tau_s'].median() * 1e3:.0f} ms median ({CYC['tail_tau_s'].min() * 1e3:.0f} to {CYC['tail_tau_s'].max() * 1e3:.0f}), "
      f"the spin is {1.5 / CYC['tail_tau_s'].median():.0f} of them long")
print(f"ripple speed over bemf-coast's: mean {CYC['speed_over_bemf_pct'].mean():+.2f} %, spread {CYC['speed_over_bemf_pct'].std():.2f} %, "
      f"ripples counted per window {CYC['ripples'].min()} to {CYC['ripples'].max()}, line prominence {CYC['line_prominence'].min():.0f} and up")

Left, how far each spin's current was still heading at the end of its window. Right, the speed counted in the spin's current ripple against the speed bemf-coast counted in the coast's voltage, spin by spin.

### Current against speed

Per rung, the settled current and its spread over the spins against the mean speed, one color per level, with the coast's own deceleration at the same speed drawn as a current through one inertia. The thing to see is whether the crosses land on the dots at both levels, and whether the two levels fall on one curve.

In [ ]:
turn = CYC[CYC["turned"] & CYC["j_kg_m2"].notna()]
RUNG = turn.groupby(["level", "duty"]).agg(n=("i_win_a", "size"), speed_rev_s=("speed_rev_s", "mean"), speed_sd=("speed_rev_s", "std"),
                                           i_ma=("i_win_a", lambda x: x.mean() * 1e3), i_sd_ma=("i_win_a", lambda x: x.std() * 1e3),
                                           gap_ma=("tail_gap_ma", "mean"), e_spin_v=("e_spin_v", "mean"), dec=("dec_at_spin_v_per_s", "mean"),
                                           j=("j_kg_m2", "mean"), j_sd=("j_kg_m2", "std"), rail_v=("rail_v", "mean")).reset_index()
RUNG["i_se_ma"] = RUNG["i_sd_ma"] / np.sqrt(RUNG["n"])
J_POOLED = float(RUNG["j"].median())                                        # one inertia, the rungs' median
RUNG["i_coast_ma"] = J_POOLED * RUNG["dec"] / KE_V ** 2 * 1e3               # the coast's drag at the rung's speed, as a current through J_POOLED
show(RUNG, n=None)
fig, ax = plt.subplots(figsize=(9, 5))
for L in LEVELS:
    g = RUNG[RUNG["level"] == L]; c = turn[turn["level"] == L]
    ax.plot(c["speed_rev_s"], c["i_win_a"] * 1e3, ".", ms=4, color=COLOR[L], alpha=0.4)                        # every spin, faint
    ax.errorbar(g["speed_rev_s"], g["i_ma"], yerr=g["i_sd_ma"], fmt="o", ms=7, color=COLOR[L], capsize=3, label=f"{L:g} V: settled current per rung, +/- sd")
    ax.plot(g["speed_rev_s"], g["i_coast_ma"], "x", ms=9, mew=1.5, color=COLOR[L], label=f"{L:g} V: the coast's drag as a current through J")
ax.set_xlabel("speed from the window's ripple (rev/s)"); ax.set_ylabel("current (mA)")
ax.set_title(f"no-load current against speed, the coast's drag through one J = {J_POOLED:.2e} kg m$^2$")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
for L in LEVELS:
    g = RUNG[RUNG["level"] == L]
    print(f"{L:g} V: {g['i_ma'].min():.1f} mA at {g['speed_rev_s'].min():.0f} rev/s to {g['i_ma'].max():.1f} mA at {g['speed_rev_s'].max():.0f} rev/s; "
          f"coast's drag as current less the measured, per rung: {np.round((g['i_coast_ma'] - g['i_ma']).to_numpy(), 1).tolist()} mA")

Dots with whiskers are the rungs, faint points the spins behind them, crosses the current the coast's drag would need at that rung's speed through the one inertia. Crosses on the dots at every rung mean the spin's current is the coast's drag and nothing else.

### The drag

The coast's deceleration at the spin's EMF, every turning coast, with each level's fitted line $a + E/\tau$ from the medians of its coasts. The thing to see is the intercept, the drag left at zero speed, and whether the two levels share a line, since the coast is the same free rotor whichever rail spun it up.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
E_line = np.linspace(0, turn["e_spin_v"].max() * 1.05, 20)
drag = {}
for L in LEVELS:
    g = turn[turn["level"] == L]
    a_med, tau_med = float(g["drag_a_v_per_s"].median()), float(g["drag_tau_s"].median())
    drag[L] = (a_med, tau_med)
    ax[0].plot(g["e_spin_v"], g["dec_at_spin_v_per_s"], "o", ms=5, color=COLOR[L], label=f"{L:g} V: deceleration at the spin's EMF")
    ax[0].plot(E_line, a_med + E_line / tau_med, "-", lw=1, color=COLOR[L], label=f"{L:g} V: a {a_med:.2f} V/s + E / {tau_med * 1e3:.0f} ms")
    ax[1].plot(g["speed_rev_s"], g["drag_a_v_per_s"], "o", ms=5, color=COLOR[L], label=f"{L:g} V: a per coast")
    ax[1].plot(g["speed_rev_s"], g["drag_tau_s"] * 1e3 / 100, "s", ms=5, mfc="none", color=COLOR[L], label=f"{L:g} V: tau per coast, in 100 ms")
ax[0].set_xlabel("back-EMF at the spin's speed (V)"); ax[0].set_ylabel("-dE/dt from the coast's fit (V/s)"); ax[0].set_title("the coast's drag against speed"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("speed the coast started from (rev/s)"); ax[1].set_ylabel("fitted term"); ax[1].set_title("the two drag terms, coast by coast"); ax[1].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
A_ALL, TAU_ALL = float(turn["drag_a_v_per_s"].median()), float(turn["drag_tau_s"].median())
print(f"pooled: a {A_ALL:.2f} V/s ({turn['drag_a_v_per_s'].quantile(0.16):.2f} to {turn['drag_a_v_per_s'].quantile(0.84):.2f}), "
      f"tau {TAU_ALL * 1e3:.0f} ms ({turn['drag_tau_s'].quantile(0.16) * 1e3:.0f} to {turn['drag_tau_s'].quantile(0.84) * 1e3:.0f}); "
      f"the Coulomb share as a current through J: {J_POOLED * A_ALL / KE_V ** 2 * 1e3:.1f} mA; "
      f"at {turn['e_spin_v'].max():.2f} V the viscous share is {turn['e_spin_v'].max() / TAU_ALL / A_ALL:.1f} times the Coulomb one")

Left, the deceleration the coast's fit gives at each spin's EMF, with each level's line through its medians. Right, the two fitted terms coast by coast against the speed the coast started from, which says whether one pair of terms describes a coast from any speed.

### One inertia

The inertia per spin against speed with a line through all of them. The thing to see is whether the line is flat within its error, which is what one inertia across the rungs looks like.

In [ ]:
J_LINE = laps.line(turn["speed_rev_s"], turn["j_kg_m2"])                  # J against speed: a flat line is one inertia
J_RUNG = RUNG["j"]
fig, ax = plt.subplots(figsize=(9, 4.5))
for L in LEVELS:
    g = turn[turn["level"] == L]; r = RUNG[RUNG["level"] == L]
    ax.plot(g["speed_rev_s"], g["j_kg_m2"] * 1e9, ".", ms=5, color=COLOR[L], alpha=0.5)
    ax.errorbar(r["speed_rev_s"], r["j"] * 1e9, yerr=r["j_sd"] * 1e9, fmt="o", ms=7, color=COLOR[L], capsize=3, label=f"{L:g} V: J per rung, +/- sd")
x = np.linspace(turn["speed_rev_s"].min(), turn["speed_rev_s"].max(), 10)
ax.plot(x, (J_LINE["intercept"] + J_LINE["slope"] * x) * 1e9, "-", color=INK, lw=1,
        label=f"line: slope {J_LINE['slope'] * 1e9:+.3f} +/- {J_LINE['slope_se'] * 1e9:.3f} nkg m$^2$ per rev/s")
ax.axhspan(J_RUNG.min() * 1e9, J_RUNG.max() * 1e9, color=MUTED, alpha=0.15, label="the rungs' spread")
ax.set_xlabel("speed from the window's ripple (rev/s)"); ax.set_ylabel("J (1e-9 kg m$^2$)"); ax.set_title("the inertia that ties each spin's current to its coast's drag")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"J {J_POOLED:.3e} kg m^2, the rungs' spread {J_RUNG.min():.3e} to {J_RUNG.max():.3e} ({(J_RUNG.max() / J_RUNG.min() - 1) * 100:.1f} %), "
      f"sd over mean across spins {turn['j_kg_m2'].std() / turn['j_kg_m2'].mean() * 100:.1f} %")
print(f"slope with speed {J_LINE['slope'] / J_POOLED * 100:+.3f} % of J per rev/s, {abs(J_LINE['slope'] / J_LINE['slope_se']):.1f} standard errors from flat; "
      f"over the speed range that is {J_LINE['slope'] * (x[-1] - x[0]) / J_POOLED * 100:+.1f} % of J")
for L in LEVELS:
    r = RUNG[RUNG["level"] == L]
    print(f"{L:g} V: J per rung {np.round(r['j'].to_numpy() * 1e9, 2).tolist()} nkg m^2 at {np.round(r['speed_rev_s'].to_numpy(), 0).tolist()} rev/s")

Rungs as dots with whiskers, spins as faint points, the line through the spins and the rungs' spread shaded. A slope within its error of zero means one inertia, and a J that climbs with speed means the drive has a loss the free coast does not.

### The results file

The settled current per rung with its speed, both levels, as the no-load current the later stages read. The two drag terms pooled over the turning coasts with their 16th to 84th percentiles, the Coulomb term as the current it costs through the inertia, the inertia with the rungs' spread as its bounds and its slope with speed as the test of one inertia, whether every window settled and the largest tail gap, the tail's time constant, and how the spin's ripple speed sits against the coast's.

In [ ]:
q = results.quantity
def pct_q(s, unit):
    return q(s.median(), lo=s.quantile(0.16), hi=s.quantile(0.84), unit=unit)

i0_by_speed = [{"level_v": float(r["level"]), "duty_pct": int(r["duty"]), "speed_rev_s": float(r["speed_rev_s"]), "speed_sd_rev_s": float(r["speed_sd"]),
                "current_a": float(r["i_ma"] / 1e3), "lo": float((r["i_ma"] - r["i_se_ma"]) / 1e3), "hi": float((r["i_ma"] + r["i_se_ma"]) / 1e3),
                "unit": "A", "n": int(r["n"])} for _, r in RUNG.iterrows()]
results.write("2-motor", "no-load-current", CAPTURE, pair=MOTOR.key,
              notes="read from the bemf-coast dataset, no capture of its own: the settled current is the mean of each spin's last 300 ms through the "
                    "current-scale chain less the cycle's own rest zero and the in-drive offset, the speed is the commutation ripple counted in the same "
                    "window over ripple_per_rev, the drag is dE/dt = -a - E/tau fitted through each coast's back-EMF over the terminal-chain scale, "
                    "and J = Ke^2 I / (a + E/tau) at the spin's own EMF with Ke from bemf-coast; i0_by_speed is one entry per level and duty, "
                    "the current's bounds its standard error over the spins; j_kg_m2 bounds are the rungs' spread",
              ke_v_s_per_rad=KE["ke_v_s_per_rad"],
              i0_by_speed=i0_by_speed,
              i0_min=q(RUNG["i_ma"].min() / 1e3, unit="A"),
              i0_max=q(RUNG["i_ma"].max() / 1e3, unit="A"),
              speed_range_rev_s=q(RUNG["speed_rev_s"].max(), lo=RUNG["speed_rev_s"].min(), hi=RUNG["speed_rev_s"].max(), unit="rev/s"),
              drag_coulomb=pct_q(turn["drag_a_v_per_s"], "V/s"),
              drag_coulomb_a=q(J_POOLED * A_ALL / KE_V ** 2, lo=J_RUNG.min() * turn["drag_a_v_per_s"].quantile(0.16) / KE_V ** 2,
                               hi=J_RUNG.max() * turn["drag_a_v_per_s"].quantile(0.84) / KE_V ** 2, unit="A"),
              drag_viscous_tau_s=pct_q(turn["drag_tau_s"], "s"),
              j_kg_m2=q(J_POOLED, lo=J_RUNG.min(), hi=J_RUNG.max(), unit="kg m^2"),
              j_slope_per_rev_s=q(J_LINE["slope"] / J_POOLED * 100, lo=(J_LINE["slope"] - J_LINE["slope_se"]) / J_POOLED * 100,
                                  hi=(J_LINE["slope"] + J_LINE["slope_se"]) / J_POOLED * 100, unit="% of J per rev/s"),
              window_settled=q(float(CYC["settled"].all()), unit=f"1 when every spin's tail asymptote sits within {TAIL_TOL_MA:g} mA of its window mean"),
              tail_gap_ma=q(CYC["tail_gap_ma"].abs().max(), lo=CYC["tail_gap_ma"].min(), hi=CYC["tail_gap_ma"].max(), unit="mA"),
              spin_tail_tau_s=pct_q(CYC["tail_tau_s"].dropna(), "s"),
              speed_over_bemf_pct=q(CYC["speed_over_bemf_pct"].mean(), lo=CYC["speed_over_bemf_pct"].min(), hi=CYC["speed_over_bemf_pct"].max(), unit="%"),
              coasts_turned=q(int(CYC["turned"].sum()), lo=0, hi=len(CYC), unit="of the spins"))

## Limitations

- The current is one sample per control tick at the crest of the PWM window, where the drive is on, and the winding current ramps through every window. A sample at the middle of the on time reads the window's mean for a ramp that is straight, and the sample-timing experiment is where the sample's place in the ramp is graded. The current here is the sampled current, and a bias of the sample against the window's mean moves every rung by the same share.
- The in-drive offset from current-scale was read at 100 % duty, with the bridge not switching. Whether the same offset holds at 15 to 60 % is not tested here, and a different one moves every rung by a constant milliamp or two.
- $K_t = K_e$ is taken as a fact of SI units. The inertia and the Coulomb current rest on $K_e$ squared, so its error from bemf-coast enters twice.
- The coast's first row is the first open-bridge row the sweep recorded, and the host's time between the spin's last row and it is not in the file. The drag fit starts at that row and does not need the spin's end, but the comparison of the spin's ripple speed to the coast's counted speed carries that gap.
- The tap offset of the terminal chain is taken to cancel in the tap difference. An offset that differs between the taps adds a constant to the back-EMF and lands in the Coulomb term.
- The rotor's temperature rises a little through a cycle and the winding's resistance with it. The drag and the inertia do not read the resistance, so this enters only through the current the drive settles at, which the coast's own drag is compared against.
- The viscous plus Coulomb model is the simplest two-term drag. A drag that goes with the square of speed, air on the rotor, would show as a curve in the deceleration against EMF, and a line fitted through it reads a tau that moves with the starting speed.
- Thirty cycles on one motor. The spread across rungs and directions is this motor's, and a second motor of the batch is read only at rest in rest-resistance.

## What earlier work established

Before this stage the bare motor of an earlier MG90 clone was characterized once at a 5.0 V rail with 300 ms spins, a first pass at the methods used here. Its data is not reused, and its conclusions bear on how this notebook is built. The no-load current rose with speed and not in a line, concave over 12 to 40 % duty. One inertia tied the drive current to the coast's deceleration across four rungs with a spread under 5 %, so no drive-only loss was needed to explain the rise, and the drag's nature, friction or iron loss, stayed open. The 300 ms spins had not settled, their tails fitted with time constants of 70 to 170 ms and asymptotes a few milliamps under the window means, which is why the spins here run 1.5 s and why the tail fit is the first check.

## Open questions

- What the drag is. Friction at the brushes and the bearings, and iron loss in the rotor's laminations, both rise with speed and the coast does not tell them apart. They separate at the teardown. A spin-down of the rotor with the magnet can removed has no field and no iron loss, so what deceleration is left is friction, and the difference is the iron. A spin-down needs the rotor spun by hand or by a second motor and its speed read on the camera, since there is no back-EMF without the field.
- Whether the 1250 Hz pattern the rest-noise experiment found in the current channel can be mistaken for the ripple. At six ripples per revolution it corresponds to about 208 rev/s, and a rung whose ripple sits near it has two lines in the band. The line's prominence per spin and the agreement with bemf-coast's speed are the check in this data, and a spin whose two speeds disagree at that one rung and nowhere else is the signature.
- Whether the sample's place in the PWM window biases the current. A capture of the same spins with the sample moved inside the window, or a free-running burst across a window at a settled speed, gives the current's ramp and the window's true mean against the crest sample.
- Whether the in-drive offset at 15 to 60 % duty is the 100 % one. The coast rows carry the current channel with the bridge open, and their reading against the cycle's torque-off zero says what the channel reads with the driver awake and no winding current, which is the offset's own signature if it is the driver's supply current.
- Whether the rise with speed is the same at both levels. The same speed reached from two rails at two duties is the same free rotor, so the current should match where the speed ranges overlap. A mismatch there is the drive, not the drag, and the sample's place in a window of a different width is the first candidate.

## Conclusion

To be written from the capture.